# Ring all-reduce from ppermute

`psum` hides the algorithm. The classic one for large buffers is the **ring all-reduce**: $N$ devices in a ring, each buffer cut into $N$ chunks, a **reduce-scatter** phase that leaves each device with one fully summed chunk, then an **all-gather** phase that circulates the summed chunks. Each phase is $N - 1$ steps of `jax.lax.ppermute` (send to your right neighbour, receive from your left).

Write `ring_all_reduce(x)`:

- `x`: `(N, L)` float32; row $i$ is device $i$'s buffer, and $N$ divides $L$. Chunk $c$ is `buf[c * L/N : (c + 1) * L/N]`.
- Simulate the devices with `jax.vmap(per_device, axis_name='ring')`. Use `jax.lax.axis_index('ring')` for a device's id $i$ and `jax.lax.ppermute(v, 'ring', perm=[(j, (j + 1) % N) for j in range(N)])` to shift data one step right.

Schedule, for $s = 0, \dots, N - 2$:

1. **Reduce-scatter step** $s$: device $i$ sends its chunk $(i - s) \bmod N$; it receives chunk $(i - 1 - s) \bmod N$ from device $i - 1$ and **adds** it to its own copy of that chunk.

After the last reduce-scatter step, device $i$ holds the full sum of chunk $(i + 1) \bmod N$.

2. **All-gather step** $s$: device $i$ sends its chunk $(i + 1 - s) \bmod N$; it receives chunk $(i - s) \bmod N$ and **overwrites** its copy.

Return `(reduced, after_scatter)`, both `(N, L)` float32: every device's buffer at the end (all rows equal to the column sums of `x`), and every device's buffer right after the reduce-scatter phase (partially reduced chunks included). Do not call `psum`.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def ring_all_reduce(x):
    """x (N, L): row i is device i's buffer. Return (reduced (N, L), after_scatter (N, L))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0], [10.0, 20.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[11.0, 22.0], [11.0, 22.0]], "dtype": "float32"}, {"__nd__": [[1.0, 22.0], [11.0, 20.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], "dtype": "float32"}, {"__nd__": [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.0, 0.0, 1.0, 1.0, 2.0, 2.0], [3.0, 3.0, 4.0, 4.0, 5.0, 5.0], [6.0, 6.0, 7.0, 7.0, 8.0, 8.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[9.0, 9.0, 12.0, 12.0, 15.0, 15.0], [9.0, 9.0, 12.0, 12.0, 15.0, 15.0], [9.0, 9.0, 12.0, 12.0, 15.0, 15.0]], "dtype": "float32"}, {"__nd__": [[0.0, 0.0, 12.0, 12.0, 10.0, 10.0], [3.0, 3.0, 4.0, 4.0, 15.0, 15.0], [9.0, 9.0, 11.0, 11.0, 8.0, 8.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-1.0, -4.0, -1.0, -1.0, -1.0, 4.0, 5.0, -1.0], [5.0, 2.0, -1.0, 0.0, 4.0, 0.0, -1.0, 4.0], [-3.0, -3.0, -5.0, -4.0, 3.0, -2.0, -4.0, -1.0], [-5.0, 2.0, 4.0, 5.0, 0.0, -3.0, -2.0, -2.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[-4.0, -3.0, -3.0, 0.0, 6.0, -1.0, -2.0, 0.0], [-4.0, -3.0, -3.0, 0.0, 6.0, -1.0, -2.0, 0.0], [-4.0, -3.0, -3.0, 0.0, 6.0, -1.0, -2.0, 0.0], [-4.0, -3.0, -3.0, 0.0, 6.0, -1.0, -2.0, 0.0]], "dtype": "float32"}, {"__nd__": [[-1.0, -4.0, -3.0, 0.0, 2.0, -1.0, 3.0, -3.0], [4.0, -2.0, -1.0, 0.0, 6.0, -1.0, 2.0, 1.0], [1.0, -5.0, -6.0, -4.0, 3.0, -2.0, -2.0, 0.0], [-4.0, -3.0, -2.0, 1.0, 3.0, -5.0, -2.0, -2.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0, 1.0, 1.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[1.0, 1.0, 1.0]], "dtype": "float32"}, {"__nd__": [[1.0, 1.0, 1.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.7945857990952586, -1.3123958096530581, 0.31208537280385784, 0.10726449474332228, -0.7562043571946244, -0.17826817824803665, 0.19453251016823672, -0.40035000311802443, -1.0244894986135822, -0.2884227199950992], [0.21270754960406757, -0.97871263693162, -0.5874662416188372, 0.5894908688256428, 1.0509949987243061, 0.7725213424192332, 0.27178241375433115, 1.1471926228241676, -0.4285551849219527, -0.10370803350669193], [-0.21758682157248416, 0.8585956100024226, -0.7607229051692318, -0.46370114178137833, 1.57944996835367, -0.47536371326098836, -1.814458971373091, 0.7313867749831723, 0.9545632419205371, -0.9027843238382685], [-0.9042635154815833, -0.6121925162693312, 0.07264520715174679, 1.1831658258162523, 0.9620311858692903, -1.3296511834144074, 1.4491553124938925, -0.7597399436112279, 0.2670857280074569, -0.06266651895658107], [-0.36425542810150174, -1.1294818151377628, 2.098100430481616, -0.01711117352290586, 1.37109057777342, -0.6688140823166369, 0.5994637368325743, -0.4719322427738545, 0.42938075615163185, 0.9932771728663493]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[-2.067984104156494, -3.174187183380127, 1.1346418857574463, 1.39910888671875, 4.207362651824951, -1.8795759677886963, 0.7004750967025757, 0.24655723571777344, 0.1979849934577942, -0.36430442333221436], [-2.067984104156494, -3.174187183380127, 1.1346418857574463, 1.39910888671875, 4.207362651824951, -1.8795759677886963, 0.7004750967025757, 0.24655723571777344, 0.1979849934577942, -0.36430442333221436], [-2.067984104156494, -3.174187183380127, 1.1346418857574463, 1.39910888671875, 4.207362651824951, -1.8795759677886963, 0.7004750967025757, 0.24655723571777344, 0.1979849934577942, -0.36430442333221436], [-2.067984104156494, -3.174187183380127, 1.1346418857574463, 1.39910888671875, 4.207362651824951, -1.8795759677886963, 0.7004750967025757, 0.24655723571777344, 0.1979849934577942, -0.36430442333221436], [-2.067984104156494, -3.174187183380127, 1.1346418857574463, 1.39910888671875, 4.207362651824951, -1.8795759677886963, 0.7004750967025757, 0.24655723571777344, 0.1979849934577942, -0.36430442333221436]], "dtype": "float32"}, {"__nd__": [[-0.7945858240127563, -1.3123958110809326, 1.1346418857574463, 1.39910888671875, 3.156367540359497, -2.652097225189209, 2.2431516647338867, -1.6320221424102783, -0.5951087474822998, 0.7048544883728027], [-0.5818783044815063, -2.2911083698272705, -0.5874662399291992, 0.5894908905029297, 4.207362651824951, -1.8795759677886963, 2.5149340629577637, -0.48482954502105713, -1.0236639976501465, 0.6011464595794678], [-0.7994651198387146, -1.4325127601623535, -1.348189115524292, 0.12578976154327393, 1.579450011253357, -0.47536370158195496, 0.7004750967025757, 0.24655723571777344, -0.06910073757171631, -0.3016378879547119], [-1.7037286758422852, -2.044705390930176, -1.2755439281463623, 1.3089555501937866, 2.5414812564849854, -1.805014967918396, 1.449155330657959, -0.7597399353981018, 0.1979849934577942, -0.36430442333221436], [-2.067984104156494, -3.174187183380127, 0.8225564956665039, 1.291844367980957, 3.912571907043457, -2.4738290309906006, 2.048619031906128, -1.2316721677780151, 0.42938074469566345, 0.9932771921157837]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[75.0, 42.0, 7.0, 13.0, 91.0, 5.0, 66.0, 70.0, 64.0, 61.0, 84.0, 28.0, 95.0, 46.0, 12.0, 64.0], [0.0, 86.0, 43.0, 93.0, 17.0, 58.0, 93.0, 74.0, 93.0, 81.0, 7.0, 88.0, 76.0, 49.0, 65.0, 32.0], [51.0, 89.0, 92.0, 18.0, 21.0, 42.0, 32.0, 96.0, 38.0, 23.0, 15.0, 18.0, 50.0, 49.0, 35.0, 64.0], [43.0, 47.0, 64.0, 74.0, 13.0, 73.0, 37.0, 24.0, 89.0, 16.0, 67.0, 21.0, 43.0, 74.0, 21.0, 35.0], [42.0, 31.0, 7.0, 17.0, 10.0, 5.0, 60.0, 27.0, 24.0, 46.0, 22.0, 30.0, 71.0, 98.0, 61.0, 0.0], [51.0, 12.0, 69.0, 3.0, 21.0, 47.0, 31.0, 56.0, 95.0, 57.0, 0.0, 90.0, 85.0, 65.0, 80.0, 45.0], [17.0, 45.0, 92.0, 15.0, 88.0, 86.0, 44.0, 32.0, 13.0, 58.0, 12.0, 1.0, 21.0, 38.0, 3.0, 56.0], [32.0, 83.0, 46.0, 89.0, 77.0, 15.0, 31.0, 82.0, 54.0, 22.0, 98.0, 33.0, 16.0, 97.0, 4.0, 25.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[311.0, 435.0, 420.0, 322.0, 338.0, 331.0, 394.0, 461.0, 470.0, 364.0, 305.0, 309.0, 457.0, 516.0, 281.0, 321.0], [311.0, 435.0, 420.0, 322.0, 338.0, 331.0, 394.0, 461.0, 470.0, 364.0, 305.0, 309.0, 457.0, 516.0, 281.0, 321.0], [311.0, 435.0, 420.0, 322.0, 338.0, 331.0, 394.0, 461.0, 470.0, 364.0, 305.0, 309.0, 457.0, 516.0, 281.0, 321.0], [311.0, 435.0, 420.0, 322.0, 338.0, 331.0, 394.0, 461.0, 470.0, 364.0, 305.0, 309.0, 457.0, 516.0, 281.0, 321.0], [311.0, 435.0, 420.0, 322.0, 338.0, 331.0, 394.0, 461.0, 470.0, 364.0, 305.0, 309.0, 457.0, 516.0, 281.0, 321.0], [311.0, 435.0, 420.0, 322.0, 338.0, 331.0, 394.0, 461.0, 470.0, 364.0, 305.0, 309.0, 457.0, 516.0, 281.0, 321.0], [311.0, 435.0, 420.0, 322.0, 338.0, 331.0, 394.0, 461.0, 470.0, 364.0, 305.0, 309.0, 457.0, 516.0, 281.0, 321.0], [311.0, 435.0, 420.0, 322.0, 338.0, 331.0, 394.0, 461.0, 470.0, 364.0, 305.0, 309.0, 457.0, 516.0, 281.0, 321.0]], "dtype": "float32"}, {"__nd__": [[75.0, 42.0, 420.0, 322.0, 321.0, 273.0, 269.0, 291.0, 250.0, 244.0, 194.0, 152.0, 132.0, 181.0, 16.0, 89.0], [75.0, 128.0, 43.0, 93.0, 338.0, 331.0, 362.0, 365.0, 343.0, 325.0, 201.0, 240.0, 208.0, 230.0, 81.0, 121.0], [126.0, 217.0, 135.0, 111.0, 21.0, 42.0, 394.0, 461.0, 381.0, 348.0, 216.0, 258.0, 258.0, 279.0, 116.0, 185.0], [169.0, 264.0, 199.0, 185.0, 34.0, 115.0, 37.0, 24.0, 470.0, 364.0, 283.0, 279.0, 301.0, 353.0, 137.0, 220.0], [211.0, 295.0, 206.0, 202.0, 44.0, 120.0, 97.0, 51.0, 24.0, 46.0, 305.0, 309.0, 372.0, 451.0, 198.0, 220.0], [262.0, 307.0, 275.0, 205.0, 65.0, 167.0, 128.0, 107.0, 119.0, 103.0, 0.0, 90.0, 457.0, 516.0, 278.0, 265.0], [279.0, 352.0, 367.0, 220.0, 153.0, 253.0, 172.0, 139.0, 132.0, 161.0, 12.0, 91.0, 21.0, 38.0, 281.0, 321.0], [311.0, 435.0, 413.0, 309.0, 230.0, 268.0, 203.0, 221.0, 186.0, 183.0, 110.0, 124.0, 37.0, 135.0, 4.0, 25.0]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(ring_all_reduce, json.loads(_TESTS), 'jax', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")